In [4]:
import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")

MY_FILENAME = "2026-07-21-data-usability-report"


In [5]:
# Load the data

COLUMNS = [
    'JURISDICTION', 'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 'STATUS_NORMALIZED', 'RECORD_TYPE_ORIGINAL', 'APN', 'STREET', 'ZIPCODE'
]

df = pd.read_parquet(
    os.path.join(MY_DATA_PATH, "processed_data", "dewey_ca_la_county_permits.parquet"),
    columns=COLUMNS
).reset_index(drop=True)

In [6]:
# Date column that prioritizes file date over permit date, but uses permit date if file date is missing

df['FILE_OR_PERMIT_DATE'] = df['FILE_DATE'].fillna(df['PERMIT_DATE'])

In [7]:
# Initialize markdown file to output

MD = """
# Data Usability Report 

For Dewey permits data, jurisdictions in Los Angeles County. No data cleaning was done for any of the variables.
"""

In [8]:
# Write report

JURISDICTIONS = sorted(df['JURISDICTION'].unique().tolist())
STATUSES = ['Active', 'Final', 'In Review', 'Inactive']
DATE_CONCEPTS = ['FILE_DATE', 'PERMIT_DATE', 'FILE_OR_PERMIT_DATE', 'FINAL_DATE']
STATUS_DC_SUMMARY = {}  # for each status/date concept, number of jurisdictions with >90% populated
USABILITY_THRESHOLD = 0.85

for j in JURISDICTIONS:
    MD += f"\n## {j} \n"
    mask = df['JURISDICTION'] == j
    count = mask.sum()
    status_not_na = (df.loc[mask, 'STATUS_NORMALIZED'].notna()).sum()
    pass_fail = "*OK*" if status_not_na/count > USABILITY_THRESHOLD else "**FAIL**"
    MD += f"  - STATUS_NORMALIZED: {status_not_na:,.0f}/{count:,.0f} ({status_not_na/count:.1%}) populated {pass_fail}\n"

    for s in STATUSES:
        if s not in STATUS_DC_SUMMARY:
            STATUS_DC_SUMMARY[s] = {}

        mask = (df['JURISDICTION'] == j) & (df['STATUS_NORMALIZED'] == s)
        count_s = mask.sum()
        MD += f"    - {s}: {count:,.0f} ({count_s/status_not_na:.1%})\n"

        for dc in DATE_CONCEPTS:
            if dc not in STATUS_DC_SUMMARY[s]:
                STATUS_DC_SUMMARY[s][dc] = 0
                
            count_dc = (df.loc[mask, dc].notna()).sum()
            pass_fail = "*OK*" if count_dc/count_s > USABILITY_THRESHOLD else "**FAIL**"
            MD += f"      - {dc}: {count_dc:,.0f} / {count_s:,.0f} ({count_dc/count_s:.1%}) populated {pass_fail}\n"
            if pass_fail == "*OK*":
                STATUS_DC_SUMMARY[s][dc] = STATUS_DC_SUMMARY[s].get(dc, 0) + 1

print("\n## Summary")
for s in STATUSES:
    MD += f"\n  - {s}:\n"
    for dc, v in STATUS_DC_SUMMARY[s].items():
        MD += f"    - {dc}: {v:,.0f} / {len(JURISDICTIONS):,.0f} usable jurisdictions\n"



## Summary


In [9]:
print(MD)


# Data Usability Report 

For Dewey permits data, jurisdictions in Los Angeles County. No data cleaning was done for any of the variables.

## Alhambra 
  - STATUS_NORMALIZED: 15,420/16,110 (95.7%) populated *OK*
    - Active: 16,110 (24.9%)
      - FILE_DATE: 3,838 / 3,838 (100.0%) populated *OK*
      - PERMIT_DATE: 3,795 / 3,838 (98.9%) populated *OK*
      - FILE_OR_PERMIT_DATE: 3,838 / 3,838 (100.0%) populated *OK*
      - FINAL_DATE: 60 / 3,838 (1.6%) populated **FAIL**
    - Final: 16,110 (24.9%)
      - FILE_DATE: 3,832 / 3,832 (100.0%) populated *OK*
      - PERMIT_DATE: 3,734 / 3,832 (97.4%) populated *OK*
      - FILE_OR_PERMIT_DATE: 3,832 / 3,832 (100.0%) populated *OK*
      - FINAL_DATE: 3,832 / 3,832 (100.0%) populated *OK*
    - In Review: 16,110 (24.1%)
      - FILE_DATE: 3,709 / 3,709 (100.0%) populated *OK*
      - PERMIT_DATE: 202 / 3,709 (5.4%) populated **FAIL**
      - FILE_OR_PERMIT_DATE: 3,709 / 3,709 (100.0%) populated *OK*
      - FINAL_DATE: 29 / 3,709 (0.8

In [10]:
with open(os.path.join(ROOT_PATH, "reports", f"{MY_FILENAME}.md"), "w") as f:
    f.write(MD)
